# LAPORAN ANALISIS DATA KATEGORIKAL
## Pengaruh Pendapatan dan Kondisi Sanitasi terhadap Status Stunting pada Balita

---

| Mata Kuliah   | Analisis Data Kategorikal dengan Python |
|---------------|-----------------------------------------|
| Topik         | Regresi Logistik & Analisis Data Biner  |
| Metode        | Point Bi-Serial, Regresi Logistik, Regresi Probit |
| Tanggal       | 8 Juni 2026                             |

---

## A. PENDAHULUAN DAN DESKRIPSI DATA

Penelitian ini bertujuan untuk menganalisis dua pertanyaan empiris:

1. Apakah terdapat pengaruh antara **pendapatan keluarga (Income)** terhadap **status stunting** pada balita?
2. Apakah terdapat pengaruh antara **kondisi sanitasi (Sanitation)** terhadap **status stunting** pada balita?

Data penelitian terdiri dari 25 observasi balita dengan variabel-variabel berikut:

| Variabel   | Skala Pengukuran | Keterangan                                              |
|------------|------------------|---------------------------------------------------------|
| Stunting   | Nominal (Biner)  | Status stunting: 1 = Stunting, 0 = Tidak Stunting       |
| Income     | Rasio            | Pendapatan keluarga per bulan dalam juta rupiah         |
| Nutrition  | Interval         | Skor asupan gizi balita (0–100)                         |
| Sanitation | Interval         | Skor kondisi sanitasi rumah tangga (0–100)              |

### Statistika Deskriptif Berdasarkan Status Stunting

| Variabel   | Status         | Mean  | Std Dev | Min  | Max  |
|------------|----------------|-------|---------|------|------|
| Income     | Stunting (1)   | 2.83  | 0.442   | 2.1  | 3.5  |
| Income     | Tidak Stunting | 7.26  | 0.642   | 6.2  | 8.2  |
| Nutrition  | Stunting (1)   | 50.70 | 2.406   | 47   | 55   |
| Nutrition  | Tidak Stunting | 78.33 | 2.795   | 74   | 83   |
| Sanitation | Stunting (1)   | 57.80 | 2.300   | 54   | 61   |
| Sanitation | Tidak Stunting | 84.07 | 2.815   | 80   | 89   |

Terdapat perbedaan rata-rata yang sangat mencolok antara kelompok stunting dan tidak stunting pada seluruh variabel, mengindikasikan potensi diskriminasi yang kuat.

### 1. Inisialisasi Library & Lingkungan Kerja

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import pointbiserialr

import statsmodels.api as sm

from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc, roc_auc_score
)

# Pengaturan parameter estetika gambar secara global
plt.rcParams.update({
    'font.family'     : 'DejaVu Sans',
    'font.size'       : 9,
    'axes.titlesize'  : 11,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 9,
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor'  : '#FFFFFF',
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'grid.linestyle'  : '--'
})

COLORS = {
    'stunting': '#E74C3C', 'normal': '#2ECC71', 'accent': '#2C3E50',
    'model_a': '#3498DB', 'model_b': '#9B59B6', 'probit': '#C39BD3'
}

print("Inisialisasi library dan konfigurasi style visualisasi berhasil.")

### 2. Memasukkan Data Penelitian

In [ ]:
data_raw = {
    'No':         list(range(1, 26)),
    'Stunting':   [1,1,1,1,1,1,1,1,1,1,
                   0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    'Income':     [2.1,2.5,3.0,3.2,2.8,3.5,3.0,2.2,2.9,3.1,
                   6.5,7.0,6.8,7.5,8.0,6.2,7.3,6.9,8.2,7.8,
                   6.7,8.1,7.4,6.6,7.9],
    'Nutrition':  [55,50,52,48,53,49,51,47,50,52,
                   75,78,80,77,79,76,82,74,81,79,
                   77,83,78,76,80],
    'Sanitation': [60,58,61,55,57,59,56,54,60,58,
                   80,82,85,83,86,81,88,80,87,84,
                   83,89,85,82,86]
}
data = pd.DataFrame(data_raw)
print(f"Total observasi : {len(data)}")
display(data.head(10))

### 3. Pembuatan Visualisasi Statistika Deskriptif

In [ ]:
# Visualisasi Analisis Deskriptif
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Analisis Deskriptif: Distribusi Variabel Berdasarkan Status Stunting', fontsize=12, fontweight='bold', color=COLORS['accent'], y=1.02)

# 1. Distribusi Stunting
counts = data['Stunting'].value_counts().sort_index()
bars = axes[0].bar(['Tidak Stunting (0)', 'Stunting (1)'],
                  counts.values,
                  color=[COLORS['normal'], COLORS['stunting']],
                  edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val} ({val/len(data)*100:.0f}%)',
                 ha='center', va='bottom', fontweight='bold', fontsize=9)
axes[0].set_title('Distribusi Status Stunting')
axes[0].set_ylabel('Jumlah Balita')
axes[0].set_ylim(0, max(counts.values) * 1.25)

# 2. Boxplot Income
bp1 = axes[1].boxplot(
    [data[data['Stunting']==0]['Income'], data[data['Stunting']==1]['Income']],
    labels=['Tidak Stunting', 'Stunting'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5)
)
bp1['boxes'][0].set_facecolor(COLORS['normal'])
bp1['boxes'][1].set_facecolor(COLORS['stunting'])
axes[1].set_title('Income per Status Stunting')
axes[1].set_ylabel('Income (Juta Rp)')

# 3. Boxplot Sanitation
bp2 = axes[2].boxplot(
    [data[data['Stunting']==0]['Sanitation'], data[data['Stunting']==1]['Sanitation']],
    labels=['Tidak Stunting', 'Stunting'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5)
)
bp2['boxes'][0].set_facecolor(COLORS['normal'])
bp2['boxes'][1].set_facecolor(COLORS['stunting'])
axes[2].set_title('Skor Sanitasi per Status Stunting')
axes[2].set_ylabel('Skor Sanitasi')

plt.tight_layout()
plt.savefig('hasil_deskriptif_stunting.png', dpi=120, bbox_inches='tight')
plt.show()

## B. PEMILIHAN MODEL

### B.1 Justifikasi Pemilihan Model

Penetapan model analisis didasarkan pada **skala pengukuran** masing-masing variabel:

- **Variabel Dependen (Y = Stunting)**: Berskala **nominal dikotomi** (0 = tidak stunting, 1 = stunting). Karena variabel respons bersifat biner, distribusi error tidak memenuhi asumsi normalitas yang dipersyaratkan dalam **Ordinary Least Squares (OLS) Regression**, sehingga regresi linear biasa **tidak layak digunakan**.

- **Variabel Independen (X₁ = Income, X₂ = Sanitation)**: Berskala **rasio** dan **interval** (data kontinu).

Kondisi ini — variabel dependen nominal biner dan variabel independen bertipe kontinu — secara statistik paling tepat dianalisis menggunakan **Regresi Logistik (Logistic Regression)**, dengan alasan sebagai berikut:

1. **Kesesuaian Distribusi**: Regresi Logistik mengasumsikan bahwa Y mengikuti distribusi Bernoulli, yang sesuai untuk variabel biner.
2. **Bounded Prediction**: Melalui fungsi *sigmoid/logit* sebagai *link function*, model memastikan nilai prediksi probabilitas berada dalam rentang [0, 1].
3. **Konsistensi dengan Ukuran Asosiasi**: Point Bi-Serial Correlation merupakan ukuran asosiasi yang relevan untuk hubungan nominal–interval/rasio, dan Regresi Logistik merupakan perluasan pemodelannya.
4. **Relevansi Odds Ratio**: Koefisien dalam regresi logistik dapat dieksponensikan menjadi *odds ratio* yang memiliki makna substantif dalam konteks kesehatan masyarakat.

### B.2 Model yang Dibangun

Dua model terpisah dibangun sesuai arahan penelitian:

- **Model A**: Pengaruh Pendapatan (Income) terhadap Status Stunting
  > `logit(P(Stunting=1)) = β₀ + β₁ · Income`

- **Model B**: Pengaruh Kondisi Sanitasi (Sanitation) terhadap Status Stunting
  > `logit(P(Stunting=1)) = β₀ + β₁ · Sanitation`

## D. HASIL DAN INTERPRETASI

### D.2 Ukuran Asosiasi — Korelasi Point Bi-Serial

| Variabel   | Koef. Point Bi-Serial (r_pb) | p-value    | Kesimpulan     |
|------------|------------------------------|------------|----------------|
| Income     | −0.9695                      | < 0.000001 | **Signifikan** |
| Nutrition  | −0.9845                      | < 0.000001 | **Signifikan** |
| Sanitation | −0.9814                      | < 0.000001 | **Signifikan** |

**Interpretasi:**
- Seluruh variabel kontinu menunjukkan korelasi **negatif sangat kuat** (mendekati −1.0) dengan status stunting.
- Nilai r_pb = −0.97 untuk Income mengindikasikan bahwa semakin tinggi pendapatan keluarga, peluang balita mengalami stunting semakin rendah secara signifikan.
- Nilai r_pb = −0.98 untuk Sanitation menunjukkan bahwa kondisi sanitasi yang lebih baik berkorelasi kuat dengan tidak adanya stunting pada balita.
- Seluruh nilai p-value jauh di bawah α = 0.05, membuktikan bahwa hubungan tersebut **signifikan secara statistik**.

In [ ]:
# Menghitung Korelasi Point Bi-Serial untuk Verifikasi
Y = data['Stunting']
print("Hasil Eksekusi Korelasi Point Bi-Serial:")
print("-" * 65)
for var in ['Income', 'Nutrition', 'Sanitation']:
    r_pb, p_val = pointbiserialr(Y, data[var])
    print(f"{var:<12}: r_pb = {r_pb:.4f} | p-value = {p_val:.6f}")

### D.3 Model A — Regresi Logistik: Income → Stunting

**Persamaan Model:**
```
g(x) = 70.9093 + (−14.6771) × Income
P(Stunting=1) = 1 / (1 + e^(−g(x)))
```

**Ringkasan Hasil:**

| Evaluasi             | Nilai        | Keterangan                        |
|----------------------|--------------|-----------------------------------|
| Pseudo R²            | 1.000        | Model menjelaskan 100% variasi Y  |
| Log-Likelihood       | ≈ 0          | Fit sempurna                      |
| LLR p-value          | 6.596 × 10⁻⁹ | **< 0.05 → SIGNIFIKAN**           |
| Koef. Income (β₁)    | −14.6771     | Arah negatif                      |
| p-value parsial β₁   | 0.999        | Tidak reliabel (perfect sep.)     |
| Odds Ratio Income    | ≈ 0.0000     | Penurunan odds stunting           |
| Akurasi Klasifikasi  | **100.00%**  | Sangat Baik                       |
| ROC-AUC              | **1.0000**   | Diskriminasi Sempurna             |

**Confusion Matrix Model A:**
```
                Pred: 0    Pred: 1
Aktual: 0  →      15          0
Aktual: 1  →       0         10
```

In [ ]:
# Menjalankan Estimasi Model A & Visualisasi Kurva Logistik
Y_A = data['Stunting']
X_A = sm.add_constant(data['Income'])

model_A  = sm.Logit(Y_A, X_A)
result_A = model_A.fit(disp=0)
print(result_A.summary())

# Plot Kurva Logistik Income
plt.figure(figsize=(9, 5))
np.random.seed(42)
for st, color, label in [(1, COLORS['stunting'], 'Stunting'), (0, COLORS['normal'], 'Tidak Stunting')]:
    subset = data[data['Stunting'] == st]
    jitter_y = subset['Stunting'] + np.random.uniform(-0.03, 0.03, len(subset))
    plt.scatter(subset['Income'], jitter_y,
                color=color, alpha=0.8, s=80, zorder=5,
                label=label, edgecolors='white', linewidth=0.8)

x_range_A  = np.linspace(data['Income'].min() - 0.5, data['Income'].max() + 0.5, 300)
x_range_A_ = sm.add_constant(pd.Series(x_range_A, name='Income'))
y_pred_A   = result_A.predict(x_range_A_)
plt.plot(x_range_A, y_pred_A, color=COLORS['model_a'],
         linewidth=2.5, label='Kurva Logistik', zorder=4)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Threshold = 0.5')
plt.title('Kurva Regresi Logistik: Income -> Status Stunting Balita', fontsize=11, fontweight='bold')
plt.xlabel('Income (Juta Rupiah)')
plt.ylabel('P(Stunting = 1)')
plt.ylim(-0.1, 1.1)
plt.legend(loc='center right', fontsize=8)

beta0_A = result_A.params['const']
beta1_A = result_A.params['Income']
eq_text_A = f'g(x) = {beta0_A:.4f} + ({beta1_A:.4f})·Income\nAUC = 1.0000'
plt.text(0.05, 0.75, eq_text_A, transform=plt.gca().transAxes, fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8, edgecolor='gray'))

plt.savefig('hasil_kurva_logistik_A.png', dpi=120, bbox_inches='tight')
plt.show()

#### Interpretasi Model A:

**1. Uji Serentak (Log-Likelihood Ratio Test)**
Nilai G² = 33.65 dengan LLR p-value = 6.60 × 10⁻⁹ (p < 0.001). Hasil uji serentak menunjukkan bahwa secara simultan **terdapat pengaruh signifikan Income terhadap status stunting** pada balita. Model ini secara keseluruhan layak digunakan sebagai model prediksi status stunting.

**2. Uji Parsial (Uji Z)**
Nilai p-value pada uji parsial (Z) sebesar 0.999 tidak dapat diinterpretasikan secara langsung karena kondisi *complete separation* menyebabkan standar error β₁ menjadi sangat besar. Namun demikian, **korelasi point bi-serial yang signifikan** (r = −0.97, p < 0.001) mengkonfirmasi bahwa Income secara individual berpengaruh nyata terhadap status stunting.

**3. Odds Ratio**
Nilai Odds Ratio untuk Income sangat kecil (≈ 0), yang mencerminkan **penurunan odds kejadian stunting yang sangat drastis** setiap kali pendapatan keluarga meningkat. Secara substantif: keluarga dengan pendapatan tinggi memiliki kapasitas finansial lebih besar untuk menyediakan makanan bergizi, layanan kesehatan preventif, dan lingkungan hidup yang lebih baik bagi balita, sehingga risiko stunting menjadi jauh lebih rendah.

**4. Ketepatan Klasifikasi**
Akurasi model sebesar **100%** menunjukkan bahwa seluruh 25 observasi berhasil diklasifikasikan dengan benar. Confusion matrix memperlihatkan 0 kesalahan klasifikasi (0 *false positive*, 0 *false negative*). Ketepatan klasifikasi ini tergolong **sempurna**, konsisten dengan temuan *complete separation* pada data.

**5. ROC-AUC**
Nilai AUC = **1.0000** mengindikasikan bahwa model memiliki kemampuan diskriminasi yang **sempurna** dalam membedakan balita yang mengalami stunting dan yang tidak. Kurva ROC menyentuh sudut kiri atas grafik secara penuh. Nilai ini secara substansial mengkonfirmasi bahwa **pendapatan keluarga merupakan prediktor determinan** terhadap status stunting balita dalam dataset ini.

In [ ]:
# Evaluasi & Cetak Evaluasi Model A
data['prob_A']       = result_A.predict(X_A)
data['pred_class_A'] = (data['prob_A'] >= 0.5).astype(int)

accuracy_A = accuracy_score(Y_A, data['pred_class_A'])
cm_A       = confusion_matrix(Y_A, data['pred_class_A'])
report_A   = classification_report(Y_A, data['pred_class_A'])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Evaluasi Performa Klasifikasi: Model A (Income -> Stunting)', fontsize=12, fontweight='bold', color=COLORS['accent'], y=1.02)

# 1. Confusion Matrix
sns.heatmap(cm_A, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['Aktual: 0', 'Aktual: 1'],
            ax=axes[0], linewidths=0.5, linecolor='white', cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix Model A')

# 2. ROC Curve
fpr_A, tpr_A, _ = roc_curve(Y_A, data['prob_A'])
roc_auc_A       = auc(fpr_A, tpr_A)
axes[1].plot(fpr_A, tpr_A, color=COLORS['model_a'], lw=2.5, label=f'Model A (AUC = {roc_auc_A:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Acak (AUC=0.5)')
axes[1].fill_between(fpr_A, tpr_A, alpha=0.15, color=COLORS['model_a'])
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Model A')
axes[1].legend(loc='lower right', fontsize=8)

# 3. Probabilitas Prediksi Bar Chart
x_idx = range(len(data))
colors_bar = [COLORS['stunting'] if s == 1 else COLORS['normal'] for s in data['Stunting']]
axes[2].bar(x_idx, data['prob_A'], color=colors_bar, alpha=0.8, edgecolor='white')
axes[2].axhline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold = 0.5')
axes[2].set_xlabel('Observasi')
axes[2].set_ylabel('P(Stunting=1)')
axes[2].set_title('Probabilitas Prediksi per Kasus')
axes[2].set_xticks(x_idx)
axes[2].set_xticklabels([str(i+1) for i in x_idx], fontsize=7)
axes[2].set_ylim(0, 1.1)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('hasil_evaluasi_model_A.png', dpi=120, bbox_inches='tight')
plt.show()

### D.1 Catatan Metodologis: *Complete Separation*

Sebelum menginterpretasikan hasil, terdapat temuan penting yang perlu dicatat: model menghasilkan **peringatan *Perfect/Complete Separation***. Kondisi ini terjadi karena terdapat **pemisahan sempurna (*complete separation*)** antara kedua kelompok dalam data:

| Kelompok           | Rentang Income (Juta Rp) | Rentang Sanitation (Skor) |
|--------------------|--------------------------|---------------------------|
| Stunting (1)       | 2.1 – 3.5                | 54 – 61                   |
| Tidak Stunting (0) | 6.2 – 8.2                | 80 – 89                   |

Tidak terdapat **overlap** sama sekali antara kedua kelompok. Konsekuensi statistiknya adalah **estimasi *Maximum Likelihood* (MLE) tidak konvergen** secara formal dan standar error koefisien menjadi sangat besar (inflated). Hal ini menyebabkan uji parsial (uji Z) menghasilkan p-value yang tidak reliabel. Namun demikian, temuan *complete separation* ini **secara substantif merupakan bukti kuat** bahwa income dan sanitasi merupakan prediktor yang sangat diskriminatif untuk status stunting.

### D.4 Model B — Regresi Logistik: Sanitation → Stunting

**Persamaan Model:**
```
g(x) = 97.2646 + (−1.3887) × Sanitation
P(Stunting=1) = 1 / (1 + e^(−g(x)))
```

**Ringkasan Hasil:**

| Evaluasi              | Nilai        | Keterangan                        |
|-----------------------|--------------|-----------------------------------|
| Pseudo R²             | 1.000        | Model menjelaskan 100% variasi Y  |
| LLR p-value           | 6.596 × 10⁻⁹ | **< 0.05 → SIGNIFIKAN**           |
| Koef. Sanitation (β₁) | −1.3887      | Arah negatif                      |
| Odds Ratio Sanitation | 0.2494       | Penurunan odds stunting 75.06%    |
| Akurasi Klasifikasi   | **100.00%**  | Sangat Baik                       |
| ROC-AUC              | **1.0000**   | Diskriminasi Sempurna             |

**Confusion Matrix Model B:**
```
                Pred: 0    Pred: 1
Aktual: 0  →      15          0
Aktual: 1  →       0         10
```

In [ ]:
# Menjalankan Estimasi Model B & Visualisasi Kurva Logistik
Y_B = data['Stunting']
X_B = sm.add_constant(data['Sanitation'])

model_B  = sm.Logit(Y_B, X_B)
result_B = model_B.fit(disp=0)
print(result_B.summary())

# Plot Kurva Logistik Sanitation
plt.figure(figsize=(9, 5))
np.random.seed(42)
for st, color, label in [(1, COLORS['stunting'], 'Stunting'), (0, COLORS['normal'], 'Tidak Stunting')]:
    subset = data[data['Stunting'] == st]
    jitter_y = subset['Stunting'] + np.random.uniform(-0.03, 0.03, len(subset))
    plt.scatter(subset['Sanitation'], jitter_y,
                color=color, alpha=0.8, s=80, zorder=5,
                label=label, edgecolors='white', linewidth=0.8)

x_range_B  = np.linspace(data['Sanitation'].min() - 2, data['Sanitation'].max() + 2, 300)
x_range_B_ = sm.add_constant(pd.Series(x_range_B, name='Sanitation'))
y_pred_B   = result_B.predict(x_range_B_)
plt.plot(x_range_B, y_pred_B, color=COLORS['model_b'],
         linewidth=2.5, label='Kurva Logistik', zorder=4)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Threshold = 0.5')
plt.title('Kurva Regresi Logistik: Sanitation -> Status Stunting Balita', fontsize=11, fontweight='bold')
plt.xlabel('Skor Kondisi Sanitasi')
plt.ylabel('P(Stunting = 1)')
plt.ylim(-0.1, 1.1)
plt.legend(loc='center right', fontsize=8)

beta0_B = result_B.params['const']
beta1_B = result_B.params['Sanitation']
eq_text_B = f'g(x) = {beta0_B:.4f} + ({beta1_B:.4f})·Sanitation\nAUC = 1.0000'
plt.text(0.05, 0.75, eq_text_B, transform=plt.gca().transAxes, fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8, edgecolor='gray'))

plt.savefig('hasil_kurva_logistik_B.png', dpi=120, bbox_inches='tight')
plt.show()

#### Interpretasi Model B:

**1. Uji Serentak (Log-Likelihood Ratio Test)**
Nilai G² = 33.65 dengan LLR p-value = 6.60 × 10⁻⁹ (p < 0.001). Hasil ini membuktikan bahwa secara simultan **terdapat pengaruh signifikan kondisi sanitasi terhadap status stunting**. Model layak digunakan untuk memodelkan status stunting berdasarkan skor sanitasi.

**2. Uji Parsial (Uji Z)**
Serupa dengan Model A, p-value parsial tidak reliabel akibat *complete separation*. Namun korelasi point bi-serial yang signifikan (r = −0.98, p < 0.001) mengkonfirmasi hubungan signifikan antara sanitasi dan stunting secara individual.

**3. Odds Ratio**
Nilai Odds Ratio = **0.2494**, mengindikasikan bahwa setiap **peningkatan skor sanitasi sebesar 1 poin** akan **menurunkan odds kejadian stunting sebesar 75.06%** (`1 − 0.2494 = 0.7506`). Secara klinis-epidemiologis, kondisi sanitasi yang buruk (jamban tidak layak, air tidak bersih, pengelolaan sampah kurang) meningkatkan paparan terhadap patogen penyebab penyakit diare dan infeksi berulang. Infeksi berulang mengganggu penyerapan nutrisi dan merupakan salah satu penyebab langsung stunting pada balita.

**4. Ketepatan Klasifikasi**
Akurasi sebesar **100%** menunjukkan bahwa kondisi sanitasi mampu membedakan seluruh balita stunting dan tidak stunting dengan tepat. Nilai precision, recall, dan F1-score seluruhnya = 1.00.

**5. ROC-AUC**
Nilai AUC = **1.0000**, menunjukkan kemampuan diskriminasi model yang sempurna. **Kondisi sanitasi merupakan prediktor determinant** yang sama kuatnya dengan pendapatan dalam membedakan status stunting balita.

In [ ]:
# Evaluasi & Cetak Evaluasi Model B
data['prob_B']       = result_B.predict(X_B)
data['pred_class_B'] = (data['prob_B'] >= 0.5).astype(int)

accuracy_B = accuracy_score(Y_B, data['pred_class_B'])
cm_B       = confusion_matrix(Y_B, data['pred_class_B'])
report_B   = classification_report(Y_B, data['pred_class_B'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Evaluasi Performa Klasifikasi: Model B (Sanitation -> Stunting)', fontsize=12, fontweight='bold', color=COLORS['accent'], y=1.02)

# 1. Confusion Matrix
sns.heatmap(cm_B, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['Aktual: 0', 'Aktual: 1'],
            ax=axes[0], linewidths=0.5, linecolor='white', cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix Model B')

# 2. ROC Curve
fpr_B, tpr_B, _ = roc_curve(Y_B, data['prob_B'])
roc_auc_B       = auc(fpr_B, tpr_B)
axes[1].plot(fpr_B, tpr_B, color=COLORS['model_b'], lw=2.5, label=f'Model B (AUC = {roc_auc_B:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Acak (AUC=0.5)')
axes[1].fill_between(fpr_B, tpr_B, alpha=0.15, color=COLORS['model_b'])
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Model B')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('hasil_evaluasi_model_B.png', dpi=120, bbox_inches='tight')
plt.show()

### D.5 Model Probit sebagai Pembanding

Regresi Probit menghasilkan nilai AIC, BIC, dan AUC yang **identik** dengan Regresi Logistik, menunjukkan kedua model memiliki performa yang setara pada data ini. Model Logistik direkomendasikan karena interpretasinya (odds ratio) lebih mudah dikomunikasikan kepada pemangku kebijakan kesehatan.

**Marginal Effect at Mean (MEM) — Probit:**
- MEM Income = −0.1249: pada nilai rata-rata income (5.245 juta), setiap kenaikan pendapatan 1 juta rupiah menurunkan peluang stunting sebesar 12.49 poin persentase.
- MEM Sanitation = −0.0265: pada nilai rata-rata skor sanitasi (72.44), setiap kenaikan skor sanitasi 1 poin menurunkan peluang stunting sebesar 2.65 poin persentase.

In [ ]:
# Model Probit A & B dan Pencetakan Marginal Effects
model_probit_A  = sm.Probit(Y_A, X_A)
result_probit_A = model_probit_A.fit(disp=0)
mfx_A = result_probit_A.get_margeff(at='mean')

print("--- PROBIT MODEL A: Income → Stunting ---")
print(result_probit_A.summary())
print("\nMarginal Effects (MEM) - Model A (Income):")
display(mfx_A.summary())

model_probit_B  = sm.Probit(Y_B, X_B)
result_probit_B = model_probit_B.fit(disp=0)
mfx_B = result_probit_B.get_margeff(at='mean')

print("\n--- PROBIT MODEL B: Sanitation → Stunting ---")
print(result_probit_B.summary())
print("\nMarginal Effects (MEM) - Model B (Sanitation):")
display(mfx_B.summary())

### Perbandingan Model (Logistik vs Probit)

| Metrik              | Logistik A | Probit A | Logistik B | Probit B |
|---------------------|------------|----------|------------|----------|
| AIC                 | 4.0000     | 4.0000   | 4.0000     | 4.0000   |
| BIC                 | 6.4378     | 6.4378   | 6.4378     | 6.4378   |
| ROC-AUC             | 1.0000     | 1.0000   | 1.0000     | 1.0000   |

In [ ]:
# Perbandingan Metrik & Plot AUC Perbandingan
data_probit_A = result_probit_A.predict(X_A)
data_probit_B = result_probit_B.predict(X_B)

auc_probit_A = roc_auc_score(Y_A, data_probit_A)
auc_probit_B = roc_auc_score(Y_B, data_probit_B)

print("\n{:<30} {:>12} {:>12}".format("Metrik", "Logistik", "Probit"))
print("-" * 56)
print("{:<30} {:>12.4f} {:>12.4f}".format("AIC - Model A (Income)", result_A.aic, result_probit_A.aic))
print("{:<30} {:>12.4f} {:>12.4f}".format("BIC - Model A (Income)", result_A.bic, result_probit_A.bic))
print("{:<30} {:>12.4f} {:>12.4f}".format("AUC - Model A (Income)", roc_auc_A, auc_probit_A))
print("-" * 56)
print("{:<30} {:>12.4f} {:>12.4f}".format("AIC - Model B (Sanitation)", result_B.aic, result_probit_B.aic))
print("{:<30} {:>12.4f} {:>12.4f}".format("BIC - Model B (Sanitation)", result_B.bic, result_probit_B.bic))
print("{:<30} {:>12.4f} {:>12.4f}".format("AUC - Model B (Sanitation)", roc_auc_B, auc_probit_B))

# Visualisasi Perbandingan AUC
plt.figure(figsize=(8, 5))
model_names = ['Logistik A (Income)', 'Probit A (Income)',
               'Logistik B (Sanitation)', 'Probit B (Sanitation)']
auc_values  = [roc_auc_A, auc_probit_A, roc_auc_B, auc_probit_B]
bar_colors  = [COLORS['model_a'], '#85C1E9', COLORS['model_b'], COLORS['probit']]

bars = plt.bar(model_names, auc_values, color=bar_colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, auc_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.title('Perbandingan Nilai AUC (Logistik vs Probit)', fontsize=11, fontweight='bold', color=COLORS['accent'])
plt.ylabel('AUC Score')
plt.ylim(0.5, 1.1)
plt.axhline(0.9, color='red', linestyle='--', linewidth=1, alpha=0.5, label='AUC = 0.9')
plt.legend(fontsize=8)

plt.savefig('hasil_perbandingan_model.png', dpi=120, bbox_inches='tight')
plt.show()

## E. KESIMPULAN

Berdasarkan seluruh tahap analisis yang telah dilakukan, dapat ditarik kesimpulan sebagai berikut:

### E.1 Pemilihan Model
**Regresi Logistik** adalah model yang paling tepat untuk menganalisis pengaruh pendapatan dan sanitasi terhadap status stunting, mengingat variabel dependen bersifat biner (nominal dikotomi) dan variabel independen berskala interval/rasio. Model ini menghasilkan probabilitas prediksi yang terbatas [0,1] dan odds ratio yang bermakna secara klinis.

### E.2 Model A — Pengaruh Pendapatan terhadap Stunting
- **Persamaan**: `g(x) = 70.9093 − 14.6771 × Income`
- Uji serentak **signifikan** (LLR p = 6.60 × 10⁻⁹), menunjukkan pendapatan berpengaruh signifikan terhadap status stunting secara keseluruhan.
- Odds Ratio mendekati 0: **setiap kenaikan pendapatan menurunkan odds stunting secara drastis**.
- Akurasi klasifikasi = **100%**, AUC = **1.0000** → prediksi sempurna.

### E.3 Model B — Pengaruh Sanitasi terhadap Stunting
- **Persamaan**: `g(x) = 97.2646 − 1.3887 × Sanitation`
- Uji serentak **signifikan** (LLR p = 6.60 × 10⁻⁹), menunjukkan sanitasi berpengaruh signifikan.
- Odds Ratio = 0.2494: **setiap kenaikan 1 skor sanitasi menurunkan odds stunting sebesar 75.06%**.
- Akurasi klasifikasi = **100%**, AUC = **1.0000** → prediksi sempurna.

### E.4 Implikasi Kebijakan
Temuan penelitian ini mengimplikasikan bahwa **intervensi terhadap stunting perlu bersifat multidimensi**:

1. **Peningkatan kesejahteraan ekonomi**: Program bantuan sosial, pemberdayaan ekonomi keluarga, dan subsidi bahan pangan bergizi dapat menurunkan prevalensi stunting secara signifikan.
2. **Perbaikan infrastruktur sanitasi**: Program sanitasi total berbasis masyarakat (STBM), pengadaan air bersih, dan pembangunan jamban layak merupakan investasi kesehatan yang terbukti efektif menekan risiko stunting.
3. **Pendekatan konvergensi**: Intervensi simultan pada faktor ekonomi dan sanitasi akan memberikan dampak penurunan stunting yang lebih besar dibandingkan intervensi tunggal.

## F. REFERENSI KODE

Seluruh analisis dilakukan menggunakan:
- **Python 3.12** dengan library: `numpy`, `pandas`, `scipy`, `statsmodels`, `scikit-learn`, `matplotlib`, `seaborn`
- Kode lengkap tersedia pada file `analisis_stunting.py`
- Visualisasi tersedia pada file `hasil_analisis_stunting.png`

---

*Laporan ini dihasilkan berdasarkan data penelitian stunting dengan 25 observasi dan menerapkan metode analisis data kategorikal sesuai modul perkuliahan.*